In [13]:
import numpy as np
import pandas as pd
import json
import os
import math

from pymatgen.core.composition import Composition
from pymatgen.core.periodic_table import Species

In [14]:
from disorder.disorder import Disorder
from disorder.entropy import Entropy
from disorder.cifreader import Read_CIF

In [15]:
elem_list=['Ac', 'Ag', 'Al', 'Am', 'As', 'At', 'Au', 'B', 'Ba', 'Be', 'Bi', 'Bk', 'Br', 'C', 'Ca',\
 'Cd', 'Ce', 'Cf', 'Cl', 'Cm', 'Co', 'Cr', 'Cs', 'Cu', 'Dy', 'Er', 'Eu', 'F', 'Fe',\
 'Fr', 'Ga', 'Gd', 'Ge', 'Hf', 'Hg', 'Ho', 'I', 'In', 'Ir', 'K', 'La', 'Li', 'Lu',\
 'Mg', 'Mn', 'Mo', 'N', 'Na', 'Nb', 'Nd', 'Ni', 'No', 'Np', 'O', 'Os', 'P', 'Pa',\
 'Pb', 'Pd', 'Pm', 'Po', 'Pr', 'Pt', 'Pu', 'Ra', 'Rb', 'Re', 'Rh', 'Ru', 'S', 'Sb', 'Sc',\
 'Se', 'Si', 'Sm', 'Sn', 'Sr', 'Ta', 'Tb', 'Tc', 'Te', 'Th', 'Ti', 'Tl', 'Tm', 'U', 'V',\
 'W', 'Xe', 'Y', 'Yb', 'Zn', 'Zr']

In [16]:
path='/Users/elenapatyukova/Documents/GitHub/ICSDClient/cifs/'
list_of_files=os.listdir(path)
len(list_of_files)

221689

In [23]:
from tqdm import tqdm
from time import time

disordered_comp={}
exc=[]
exc_large=[]
exc_h=[]
exc_form=[]
errors=[]
errors_large=[]

t1=time()
limit_low=0
limit_high=len(list_of_files)
for i in tqdm(range(limit_low,limit_high)):
    file=list_of_files[i]
    h_switch=0
    form_switch=0
    try:
        cif=Read_CIF(file=path+file)
        composition=Composition(cif.read_formula).as_dict()
        for el in composition.keys():
            if(el not in elem_list):
                exc_h.append(file)
                h_switch=1
                break
                
        o=cif.orbits()
        form_el=composition.keys()
        struct_el=[]
        for name in o['atom_site_type_symbol'].values:
            struct_el.append(str(Species(name).element))
        struct_el=set(struct_el)
        if(struct_el!=form_el):
            form_switch=1
            exc_form.append(file)
                
        if(h_switch==0 and form_switch==0):
            dis=Disorder(file=path+file, radius_file='data/all_radii.csv')
            compound={}
            compound['formula']=dis.material.read_formula
            compound['ICSD_ID']=dis.material.read_id
            compound['group_num']=dis.material.space_group
            compound['Z']=dis.material.z

            # print(VPorbits,nan_value_rad)
            if(len(dis.positions)<501):
                orbits=dis.classify()
                compound=compound|orbits.to_dict()
                ent=Entropy(orbits,compound['formula'],compound['Z'])
                compound['mixing_entropy']=ent.mixing_entropy()
                compound['conf_entropy']=ent.configurational_entropy()
                disordered_comp[file]=compound
            else:
                exc_large.append(file)
                errors_large.append(list(set(dis.return_errors())))
                                    
    except:
        exc.append(file)
        errors.append(list(set(dis.return_errors())))
        
t2=time()
print(t2-t1)

100%|██████████| 20/20 [00:01<00:00, 13.20it/s]

1.5211670398712158


In [16]:
with open('disorder_results.json', 'w') as fp:
    json.dump(disordered_comp, fp, indent=2)

In [17]:
err=pd.DataFrame()
err['id']=exc_h
err.to_csv('H_files.csv')

err=pd.DataFrame()
err['id']=exc_form
err.to_csv('incomplete_structure.csv')

err=pd.DataFrame()
err['id']=exc
err['errors']=errors
err.to_csv('excluded_files.csv')

err=pd.DataFrame()
err['id']=exc_large
err['errors']=errors_large
err.to_csv('files_number_of_sites_larger_500.csv')

In [22]:
print(len(disordered_comp))
print(len(exc))
print(len(exc_large))
print(len(exc_h))
print(len(exc_form))

8
1
0
1
0
